# Train a Flatmate RL Action Policy with TRL

This notebook connects to the Hugging Face Space endpoint, collects rollout examples over OpenEnv websocket sessions, and fine-tunes a small causal language model to emit Flatmate RL JSON actions. The training path uses TRL `SFTTrainer`, which is the most stable starting point for this mixed natural-language plus structured-tool action space.

Endpoint used here: `https://huggingface.co/spaces/kushalExplores/flatmate_rl`.

In [ ]:
# Install notebook dependencies. Restart the kernel after this cell if Colab/Jupyter asks you to.
%pip install -q "trl>=0.23.0" "transformers>=4.46.0" accelerate datasets peft websockets huggingface_hub matplotlib pandas

In [ ]:
from __future__ import annotations

import asyncio
import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import websockets
from datasets import Dataset

SPACE_HTTP_URL = "https://kushalexplores-flatmate-rl.hf.space"
SCENARIOS = [
    "task_visit_single",
    "task_visit_single_hidden_flex",
    "task_visit_multi",
    "task_visit_single_seller_followup",
]

def ws_url_from_http(base_url: str) -> str:
    parsed = urlparse(base_url.rstrip("/"))
    scheme = "wss" if parsed.scheme == "https" else "ws"
    return f"{scheme}://{parsed.netloc}/ws"

SPACE_WS_URL = ws_url_from_http(SPACE_HTTP_URL)
SPACE_WS_URL

## Endpoint Client

OpenEnv's plain HTTP `/reset` and `/step` endpoints are stateless. Use `/ws` for multi-step episodes because the websocket session keeps one environment instance alive across reset and step calls.

In [ ]:
class FlatmateEndpoint:
    def __init__(self, ws_url: str = SPACE_WS_URL, timeout_s: float = 120.0):
        self.ws_url = ws_url
        self.timeout_s = timeout_s

    async def __aenter__(self):
        self.ws = await websockets.connect(self.ws_url, open_timeout=self.timeout_s, ping_timeout=self.timeout_s)
        return self

    async def __aexit__(self, exc_type, exc, tb):
        try:
            await self.ws.send(json.dumps({"type": "close"}))
        finally:
            await self.ws.close()

    async def _send(self, payload: dict[str, Any]) -> dict[str, Any]:
        await self.ws.send(json.dumps(payload))
        raw = await asyncio.wait_for(self.ws.recv(), timeout=self.timeout_s)
        message = json.loads(raw)
        if message.get("type") == "error":
            raise RuntimeError(message.get("data", message))
        data = message["data"]
        obs = data.get("observation", {})
        obs["reward"] = data.get("reward")
        obs["done"] = data.get("done", False)
        return obs

    async def reset(self, scenario_id: str, seed: int | None = None) -> dict[str, Any]:
        data: dict[str, Any] = {"scenario_id": scenario_id}
        if seed is not None:
            data["seed"] = seed
        return await self._send({"type": "reset", "data": data})

    async def step(self, action: dict[str, Any]) -> dict[str, Any]:
        return await self._send({"type": "step", "data": action})

async def smoke_test_endpoint():
    async with FlatmateEndpoint() as env:
        obs = await env.reset("task_visit_single", seed=1)
        print(obs["scenario_id"], obs["status"])
        print(obs.get("last_user_message") or obs.get("current_user_request"))

await smoke_test_endpoint()

## Rollout Policy for Data Collection

This heuristic is intentionally simple. It produces valid-looking action examples from endpoint observations; after SFT, replace it with model generation and keep the same evaluator.

In [ ]:
def tool_names(obs: dict[str, Any]) -> list[str]:
    return [str(t.get("tool", t.get("tool_name", ""))) for t in obs.get("tool_trace", [])]

def action_policy(obs: dict[str, Any]) -> dict[str, Any] | None:
    tools = tool_names(obs)
    phase = obs.get("phase", "buyer")
    remaining = set(obs.get("remaining_required_fields", []))
    scenario_id = obs.get("scenario_id", "task_visit_single")

    if phase == "seller" and not obs.get("seller_profile_stored"):
        if remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share the household dietary setup, who the flat is for, and available visit time slots."}
        return {"action_type": "tool_call", "tool_name": "store_seller_details", "tool_arguments": {}}

    if not obs.get("buyer_profile_stored"):
        if "diet" in remaining and "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference and visit availability."}
        if "diet" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference."}
        if "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your visit availability."}
        return {"action_type": "tool_call", "tool_name": "store_user_details", "tool_arguments": {}}

    if "search_posts" not in tools:
        return {"action_type": "tool_call", "tool_name": "search_posts", "tool_arguments": {}}

    post_ids = ["post_031", "post_052"] if scenario_id == "task_visit_multi" else ["post_031"]
    if "match_location_preference" not in tools:
        return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": post_ids}}
    if "get_commute_time" not in tools:
        return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": post_ids}}
    if "check_calendar_slots" not in tools:
        return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": post_ids}}
    if "shortlist" not in tools:
        return {"action_type": "tool_call", "tool_name": "shortlist", "tool_arguments": {"post_ids": post_ids}}
    if "contact_poster" not in tools:
        return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": post_ids[0], "time_text": "tomorrow 7pm"}}
    if "book_viewing" not in tools:
        return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": post_ids[0], "time_text": "tomorrow 7pm"}}

    return None

def flatten_observation(obs: dict[str, Any]) -> str:
    visible = {
        "scenario_id": obs.get("scenario_id"),
        "phase": obs.get("phase"),
        "status": obs.get("status"),
        "last_user_message": obs.get("last_user_message"),
        "current_user_request": obs.get("current_user_request"),
        "available_tools": obs.get("available_tools", []),
        "remaining_required_fields": obs.get("remaining_required_fields", []),
        "prerequisites_satisfied": obs.get("prerequisites_satisfied", {}),
        "recent_tool_calls": obs.get("recent_tool_calls", []),
        "last_tool_result": obs.get("last_tool_result", {}),
        "violations": obs.get("violations", []),
        "booked_visits": obs.get("booked_visits", []),
        "feedback_summary": obs.get("feedback_summary", ""),
    }
    return json.dumps(visible, ensure_ascii=False, sort_keys=True)

def make_training_text(obs: dict[str, Any], action: dict[str, Any]) -> str:
    return (
        "You are a broker policy for the Flatmate RL environment. "
        "Given an observation, return exactly one JSON action.\n\n"
        f"Observation:\n{flatten_observation(obs)}\n\n"
        f"Action:\n{json.dumps(action, ensure_ascii=False, sort_keys=True)}"
    )

In [ ]:
@dataclass
class RolloutConfig:
    episodes: int = 16
    max_steps: int = 20
    seed: int = 7

async def collect_rollouts(config: RolloutConfig = RolloutConfig()) -> list[dict[str, Any]]:
    rng = random.Random(config.seed)
    rows: list[dict[str, Any]] = []
    for episode_idx in range(config.episodes):
        scenario_id = rng.choice(SCENARIOS)
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=config.seed + episode_idx)
            total_reward = 0.0
            for step_idx in range(config.max_steps):
                action = action_policy(obs)
                if action is None or obs.get("done"):
                    break
                rows.append({
                    "text": make_training_text(obs, action),
                    "scenario_id": scenario_id,
                    "step": step_idx,
                    "action": json.dumps(action, sort_keys=True),
                })
                obs = await env.step(action)
                total_reward += float(obs.get("reward") or obs.get("step_reward") or 0.0)
                if obs.get("done"):
                    break
        print(f"episode={episode_idx:02d} scenario={scenario_id} rows={len(rows)} total_reward={total_reward:.2f}")
    return rows

rows = await collect_rollouts(RolloutConfig(episodes=16, max_steps=20))
dataset = Dataset.from_list(rows)
dataset

In [ ]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "flatmate-rl-action-policy"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=1536,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-5,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

train_result = trainer.train()
train_log_history = trainer.state.log_history
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
train_result

## Training Log

Plot the logged training loss over optimizer steps.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

log_path = Path(OUTPUT_DIR) / "train_log_history.json"
log_path.parent.mkdir(parents=True, exist_ok=True)
log_path.write_text(json.dumps(train_log_history, indent=2))

def plot_training_log(log_history, title: str = "SFT training loss"):
    rows = [row for row in log_history if "loss" in row and "step" in row]
    if not rows:
        print("No loss rows found in trainer.state.log_history yet.")
        return None
    df = pd.DataFrame(rows)
    ax = df.plot(x="step", y="loss", marker="o", figsize=(7, 4), title=title)
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("loss")
    ax.grid(True, alpha=0.3)
    plt.show()
    return df

train_log_df = plot_training_log(train_log_history)
train_log_df.tail() if train_log_df is not None else None

In [ ]:
import torch

def prompt_from_observation(obs: dict[str, Any]) -> str:
    return (
        "You are a broker policy for the Flatmate RL environment. "
        "Given an observation, return exactly one JSON action.\n\n"
        f"Observation:\n{flatten_observation(obs)}\n\nAction:\n"
    )

def _first_balanced_json(text: str) -> str:
    start = text.find("{")
    if start == -1:
        raise ValueError(f"No JSON object found in generation: {text!r}")
    depth = 0
    in_string = False
    escape = False
    for index, char in enumerate(text[start:], start=start):
        if escape:
            escape = False
            continue
        if char == "\\" and in_string:
            escape = True
            continue
        if char == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return text[start : index + 1]
    raise ValueError(f"Unterminated JSON object in generation: {text!r}")

def normalize_action(action: dict[str, Any]) -> dict[str, Any]:
    if action.get("action_type") == "assistant_message" and str(action.get("assistant_message", "")).strip():
        return {
            "action_type": "assistant_message",
            "assistant_message": str(action["assistant_message"]),
        }
    if action.get("action_type") == "tool_call" and str(action.get("tool_name", "")).strip():
        tool_arguments = action.get("tool_arguments", {})
        return {
            "action_type": "tool_call",
            "tool_name": str(action["tool_name"]),
            "tool_arguments": tool_arguments if isinstance(tool_arguments, dict) else {},
        }
    raise ValueError(f"Invalid action shape: {action!r}")

def parse_action(text: str) -> dict[str, Any]:
    return normalize_action(json.loads(_first_balanced_json(text)))

def heuristic_policy(obs: dict[str, Any]) -> dict[str, Any]:
    action = action_policy(obs)
    if action is None:
        return {"action_type": "assistant_message", "assistant_message": "Could you confirm the details needed for scheduling?"}
    return action

generation_stats = {"fallbacks": 0, "parse_errors": []}


def generate_action(obs: dict[str, Any], log_bad: bool = False) -> dict[str, Any]:
    # Prefix the completion with "{" so the model only has to finish a JSON object.
    prompt = prompt_from_observation(obs) + "{"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.generation_config.do_sample = False
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion_tail = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    completion = "{" + completion_tail
    try:
        return parse_action(completion)
    except Exception as exc:
        generation_stats["fallbacks"] += 1
        if len(generation_stats["parse_errors"]) < 5:
            generation_stats["parse_errors"].append({"error": str(exc), "completion": completion[:300]})
        if log_bad:
            print(f"bad generation, using fallback: {exc}")
            print("raw completion:", repr(completion[:300]))
        return heuristic_policy(obs)

async def evaluate_policy(policy_fn, label: str, scenarios=SCENARIOS, seeds=(123, 124), max_steps: int = 20, verbose: bool = False):
    rows = []
    for scenario_id in scenarios:
        for seed in seeds:
            before_fallbacks = generation_stats["fallbacks"] if "generation_stats" in globals() else 0
            async with FlatmateEndpoint() as env:
                obs = await env.reset(scenario_id, seed=seed)
                total_reward = 0.0
                steps = 0
                for step_idx in range(max_steps):
                    action = policy_fn(obs)
                    if verbose:
                        print(label, scenario_id, seed, step_idx, action)
                    obs = await env.step(action)
                    steps = step_idx + 1
                    total_reward += float(obs.get("reward") or obs.get("step_reward") or 0.0)
                    if obs.get("done"):
                        break
                after_fallbacks = generation_stats["fallbacks"] if "generation_stats" in globals() else before_fallbacks
                rows.append({
                    "policy": label,
                    "scenario_id": scenario_id,
                    "seed": seed,
                    "total_reward": total_reward,
                    "done": bool(obs.get("done")),
                    "bookings": len(obs.get("booked_visits", [])),
                    "violations": len(obs.get("violations", [])),
                    "steps": steps,
                    "fallbacks": after_fallbacks - before_fallbacks,
                })
    return rows

def raw_generate_action_text(obs: dict[str, Any]) -> str:
    prompt = prompt_from_observation(obs) + "{"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.generation_config.do_sample = False
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return "{" + tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

async def sanity_check_generations(limit: int = 4):
    rows = []
    for scenario_id in SCENARIOS[:limit]:
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=321)
            raw = raw_generate_action_text(obs)
            try:
                parsed = parse_action(raw)
                ok = True
            except Exception as exc:
                parsed = str(exc)
                ok = False
            rows.append({"scenario_id": scenario_id, "json_ok": ok, "raw": raw[:240], "parsed_or_error": parsed})
    return pd.DataFrame(rows)

async def run_inference_each_task(policy_fn, label: str, seed: int = 321, max_steps: int = 20):
    rows = []
    for scenario_id in SCENARIOS:
        print(f"\n=== {label}: {scenario_id} ===")
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=seed)
            total_reward = 0.0
            steps = 0
            before_fallbacks = generation_stats["fallbacks"]
            for step_idx in range(max_steps):
                action = policy_fn(obs)
                print(f"step={step_idx:02d} action={action}")
                obs = await env.step(action)
                steps = step_idx + 1
                total_reward += float(obs.get("reward") or obs.get("step_reward") or 0.0)
                if obs.get("done"):
                    break
            result = {
                "policy": label,
                "scenario_id": scenario_id,
                "seed": seed,
                "total_reward": total_reward,
                "done": bool(obs.get("done")),
                "bookings": len(obs.get("booked_visits", [])),
                "violations": len(obs.get("violations", [])),
                "steps": steps,
                "fallbacks": generation_stats["fallbacks"] - before_fallbacks,
            }
            print("result=", result)
            rows.append(result)
    return pd.DataFrame(rows)

generation_stats = {"fallbacks": 0, "parse_errors": []}
generation_sanity_df = await sanity_check_generations()
generation_sanity_df

per_task_inference_df = await run_inference_each_task(generate_action, "sft_trained")
print("inference_generation_stats", generation_stats)
if generation_stats["fallbacks"] > 0:
    print("WARNING: trained model produced malformed JSON and used fallback. Retrain with the updated lower LR/epochs before trusting eval_df.")
per_task_inference_df

if generation_stats["fallbacks"] == 0:
    heuristic_eval = await evaluate_policy(heuristic_policy, "heuristic")
    trained_eval = await evaluate_policy(generate_action, "sft_trained")
    eval_rows = heuristic_eval + trained_eval
    eval_df = pd.DataFrame(eval_rows)
else:
    eval_df = pd.DataFrame()
print("generation_stats", generation_stats)
eval_df

## Performance Comparison

Compare heuristic rollout behavior against the trained SFT policy on the same scenarios and seeds.

In [ ]:
def plot_policy_comparison(eval_df, title: str = "SFT policy comparison"):
    summary = (
        eval_df.groupby("policy", as_index=True)
        .agg(
            avg_reward=("total_reward", "mean"),
            completion_rate=("done", "mean"),
            avg_bookings=("bookings", "mean"),
            avg_violations=("violations", "mean"),
            avg_steps=("steps", "mean"),
        )
        .sort_index()
    )
    axes = summary[["avg_reward", "completion_rate", "avg_bookings", "avg_violations"]].plot(
        kind="bar",
        subplots=True,
        layout=(2, 2),
        figsize=(10, 7),
        legend=False,
        title=title,
    )
    for ax in axes.ravel():
        ax.grid(axis="y", alpha=0.3)
        ax.set_xlabel("")
    plt.tight_layout()
    plt.show()
    return summary

comparison_summary = plot_policy_comparison(eval_df)
comparison_summary

In [ ]:
# Optional: upload the trained adapter/model to the Hub.
# from huggingface_hub import notebook_login
# notebook_login()
# trainer.push_to_hub("flatmate-rl-action-policy")